# 크립토 강화학습 포트폴리오 운용 (PPO)

KOSPI `rl_trading_colab.ipynb`의 크립토 이식 버전.

**사전 조건**: `crypto_hybrid_model_colab.ipynb` 실행 후 Drive에 `crypto_rl_embeddings.h5` 존재

| 구성요소 | 내용 |
|----------|------|
| **State** | 종목별(44개) 임베딩(44×64) + 현재 비중(44) = 2860-dim |
| **Action** | 종목별 목표 포트폴리오 비중(44, Softmax 정규화) |
| **Reward** | 30분 포트폴리오 수익률(`RetTarget_6b`) − 거래비용(0.1%) |
| **알고리즘** | PPO (Stable-Baselines3) |

KOSPI 버전과의 차이:
- DTW 클러스터링 없이 44개 심볼을 개별 자산으로 직접 운용 (클러스터링은 추후 과제로 보류)
- 스텝 단위: 일봉 → 30분봉(6봉), 24/7 시장이라 거래일 개념 없음
- 리워드: `RetTarget_6b`가 이미 forward-looking(그 시점부터 30분 뒤 수익률)이라 KOSPI처럼 다음 스텝을 따로 조회할 필요 없음
- **타임스텝 정렬**: 심볼마다 5분봉 backfill 시작 시각이 달라 초 단위로는 30분 임베딩 샘플이 서로 어긋남(위상 차이) → `ts // 1800`(30분 버킷)으로 정렬해서 심볼 간 상태를 맞춤

> 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행 (PPO 자체는 CPU 사용, GPU는 불필요하지만 기본 설정 유지)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow

In [ ]:
import warnings
import glob
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams['axes.unicode_minus'] = False

DRIVE_ROOT = Path('/content/drive/MyDrive/졸업프로젝트')
DATA_DIR   = DRIVE_ROOT / 'data'
CRYPTO_DIR = DATA_DIR / 'crypto'
FEAT_DIR   = CRYPTO_DIR / 'features_5m'
MODEL_DIR  = DRIVE_ROOT / 'models'

RL_EMB_H5  = CRYPTO_DIR / 'crypto_rl_embeddings.h5'

EMB_DIM    = 64
TC         = 0.001    # 거래비용 (Binance taker fee 기준, KOSPI와 동일하게 가정)
BUCKET_SEC = 1800     # 30분 버킷 — 아래 1절 참고

# RL 학습/평가 기간 (하이브리드 노트북의 RL_START=2025-01-01 이후 구간을 분리)
RL_TRAIN_START = '2025-01-01'
RL_TRAIN_END   = '2025-12-31'
RL_TEST_START  = '2026-01-01'
RL_TEST_END    = '2099-12-31'   # 열린 구간 — 백필된 최신 데이터까지 전부 포함

assert RL_EMB_H5.exists(), 'crypto_rl_embeddings.h5 없음 — crypto_hybrid_model_colab.ipynb 먼저 실행'
print('경로 확인 완료')

## 1. RL 임베딩 로드 & 인덱스 구축

심볼마다 5분봉 backfill 시작 시각이 달라 30분 간격 샘플의 초 단위 timestamp가 서로 어긋난다(위상 차이). `ts // BUCKET_SEC`로 30분 버킷에 매핑하면 심볼별로 정확히 버킷당 1개씩 떨어져(STRIDE=6이 정확히 1버킷과 같으므로 충돌 없음) 서로 다른 위상이라도 같은 30분 창으로 정렬된다.

In [ ]:
print('crypto_rl_embeddings.h5 로드 중...')
with h5py.File(RL_EMB_H5, 'r') as f:
    symbols_raw = f['symbols'][:]
    ts_raw      = f['timestamps'][:]     # epoch seconds (UTC)
    embs_all    = f['embeddings'][:]     # (N, 64)

symbols_list = [s.decode() for s in symbols_raw]
ts_list      = ts_raw.astype(np.int64).tolist()

# (symbol, 30분 버킷) → 임베딩 인덱스
emb_index  = {}
sym_ts_set = {}   # 심볼별로 임베딩 추출에 실제 쓰인 초 단위 timestamp 집합 (수익률 정확 매칭용)
for i, (s, t) in enumerate(zip(symbols_list, ts_list)):
    emb_index[(s, t // BUCKET_SEC)] = i
    sym_ts_set.setdefault(s, set()).add(t)

symbols  = sorted(set(symbols_list))
N_ASSETS = len(symbols)
print(f'임베딩: {embs_all.shape} | 심볼: {N_ASSETS}개 | 인덱스: {len(emb_index):,}개')

## 2. 30분 수익률(RetTarget_6b) 로드

In [ ]:
print('features_5m parquet에서 RetTarget_6b 로드 중...')
files = sorted(glob.glob(str(FEAT_DIR / '*.parquet')))
assert files, f'피처 parquet 없음: {FEAT_DIR}'

rl_start_ts = pd.Timestamp(RL_TRAIN_START, tz='UTC')
ret_index   = {}

for fp in files:
    sym = Path(fp).stem
    valid_ts = sym_ts_set.get(sym)
    if not valid_ts:
        continue
    df = pd.read_parquet(fp, columns=['datetime', 'RetTarget_6b'])
    df = df[df['datetime'] >= rl_start_ts]
    epoch = df['datetime'].values.astype('int64') // 10**9   # ts_list와 동일 단위(초)
    y     = df['RetTarget_6b'].values
    # 임베딩 추출에 실제 쓰인 timestamp만 매칭 — 같은 버킷 내 다른 5분봉과 섞이지 않도록
    for t, r in zip(epoch, y):
        t = int(t)
        if t in valid_ts and not np.isnan(r):
            ret_index[(sym, t // BUCKET_SEC)] = float(r)

all_buckets = sorted(set(t // BUCKET_SEC for t in ts_list))
print(f'수익률 인덱스: {len(ret_index):,}개 | 전체 30분 버킷: {len(all_buckets):,}개')

## 3. CryptoPortfolioEnv (Gymnasium)

In [ ]:
class CryptoPortfolioEnv(gym.Env):
    """
    크립토 44종목 개별 자산 기반 포트폴리오 운용 환경 (클러스터링 없음)

    State:  종목별 임베딩(N×64) + 현재 비중(N) → N*64+N dim
    Action: 종목별 목표 포트폴리오 비중 (N-dim, Softmax 정규화)
    Reward: 30분 포트폴리오 수익률(RetTarget_6b, 이미 forward-looking) − 거래비용
    스텝 = 30분 버킷(BUCKET_SEC) 단위 — emb_index/ret_index 모두 (symbol, bucket) 키.
    """
    metadata = {'render_modes': []}

    def __init__(self, buckets, symbols, emb_index, embs_all, ret_index,
                 emb_dim=64, tc=0.001):
        super().__init__()
        self.buckets    = buckets
        self.symbols    = symbols
        self.n_assets   = len(symbols)
        self.emb_index  = emb_index
        self.embs_all   = embs_all
        self.ret_index  = ret_index
        self.emb_dim    = emb_dim
        self.tc         = tc

        obs_dim = self.n_assets * emb_dim + self.n_assets
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space      = spaces.Box(-3.0, 3.0, shape=(self.n_assets,), dtype=np.float32)

    def _asset_emb(self, bucket):
        """종목별 임베딩 — 해당 버킷에 데이터 없으면 0-vector(상장 전/상폐 등)"""
        embs = []
        for sym in self.symbols:
            idx = self.emb_index.get((sym, bucket))
            embs.append(self.embs_all[idx] if idx is not None else np.zeros(self.emb_dim, dtype=np.float32))
        return np.concatenate(embs)  # (N*64,)

    def _asset_ret(self, bucket):
        """종목별 30분 수익률 — 없으면 0(그 자산은 보유해도 손익 없음으로 취급)"""
        return np.array([self.ret_index.get((sym, bucket), 0.0) for sym in self.symbols], dtype=np.float32)

    def _get_obs(self):
        b = self.buckets[self.t]
        return np.concatenate([self._asset_emb(b), self.weights]).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t               = 0
        self.weights         = np.ones(self.n_assets, dtype=np.float32) / self.n_assets
        self.portfolio_value = 1.0
        self.value_history   = [1.0]
        return self._get_obs(), {}

    def step(self, action):
        # Softmax로 비중 정규화
        action  = np.exp(action - action.max())
        action  = action / action.sum()

        # 거래비용
        tc_cost = self.tc * np.abs(action - self.weights).sum()

        # RetTarget_6b는 현재 버킷 자체에 forward 수익률이 이미 인코딩되어 있음
        b          = self.buckets[self.t]
        asset_rets = self._asset_ret(b)
        port_ret   = float((action * asset_rets).sum())

        self.portfolio_value *= (1 + port_ret - tc_cost)
        self.weights           = action.copy()
        self.value_history.append(self.portfolio_value)
        self.t                += 1

        done = (self.t >= len(self.buckets) - 1)
        reward = float(port_ret - tc_cost)

        info = {'portfolio_value': self.portfolio_value, 'port_ret': port_ret}
        return self._get_obs(), reward, done, False, info

In [ ]:
def bucket_to_ts(b):
    return pd.Timestamp(b * BUCKET_SEC, unit='s', tz='UTC')

def bucket_to_date(b):
    return bucket_to_ts(b).strftime('%Y-%m-%d')

train_buckets = [b for b in all_buckets if RL_TRAIN_START <= bucket_to_date(b) <= RL_TRAIN_END]
test_buckets  = [b for b in all_buckets if RL_TEST_START  <= bucket_to_date(b) <= RL_TEST_END]
print(f'학습 기간: {bucket_to_ts(train_buckets[0])} ~ {bucket_to_ts(train_buckets[-1])} ({len(train_buckets):,}스텝)')
print(f'테스트 기간: {bucket_to_ts(test_buckets[0])} ~ {bucket_to_ts(test_buckets[-1])} ({len(test_buckets):,}스텝)')

env_kwargs = dict(
    symbols=symbols, emb_index=emb_index, embs_all=embs_all,
    ret_index=ret_index, emb_dim=EMB_DIM, tc=TC,
)

train_env = Monitor(CryptoPortfolioEnv(train_buckets, **env_kwargs))
test_env  = CryptoPortfolioEnv(test_buckets, **env_kwargs)

print('\n환경 유효성 검사...')
check_env(CryptoPortfolioEnv(train_buckets[:100], **env_kwargs), warn=True)
print('OK')

## 4. PPO 학습

In [ ]:
%%time
eval_env = Monitor(CryptoPortfolioEnv(train_buckets[-2000:], **env_kwargs))  # 최근 ~41일(30분×2000스텝)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=str(MODEL_DIR / 'crypto_eval'),   # KOSPI best_model.zip과 경로 분리
    log_path=str(CRYPTO_DIR / 'crypto_rl_logs'),
    eval_freq=10_000,
    deterministic=True,
    verbose=1,
)

ppo = PPO(
    'MlpPolicy',
    train_env,
    learning_rate=3e-4,
    n_steps=1024,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,        # 탐험 유도
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs=dict(net_arch=[512, 256]),  # obs_dim이 커서(2860) KOSPI보다 한 단 넓게
    verbose=1,
    device='cpu',          # PPO는 CPU가 더 빠른 경우 많음
    seed=42,
)

ppo.learn(
    total_timesteps=300_000,
    callback=eval_callback,
    progress_bar=True,
)

ppo.save(str(MODEL_DIR / 'crypto_ppo_portfolio'))
print('PPO 모델 저장 완료: crypto_ppo_portfolio.zip')

## 5. 백테스트 & 성과 평가

In [ ]:
def run_backtest(env, model, deterministic=True):
    """환경에서 에피소드 1회 실행 → 포트폴리오 가치 곡선 반환"""
    obs, _ = env.reset()
    done   = False
    values = [1.0]
    while not done:
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, _, done, _, info = env.step(action)
        values.append(info['portfolio_value'])
    return np.array(values)


def equal_weight_backtest(env):
    """등가중 벤치마크: 매 스텝 44종목에 균등 배분"""
    obs, _ = env.reset()
    done   = False
    values = [1.0]
    action = np.ones(env.n_assets, dtype=np.float32) / env.n_assets
    while not done:
        obs, _, done, _, info = env.step(action)
        values.append(info['portfolio_value'])
    return np.array(values)


def compute_perf(values, steps_per_year=48 * 365):
    """성과 지표 계산 — 30분봉·24/7 시장 기준 연환산(48스텝/일)"""
    rets       = np.diff(values) / values[:-1]
    total_ret  = values[-1] / values[0] - 1
    annual_ret = (1 + total_ret) ** (steps_per_year / len(rets)) - 1
    sharpe     = rets.mean() / (rets.std() + 1e-8) * np.sqrt(steps_per_year)
    max_dd     = ((values / np.maximum.accumulate(values)) - 1).min()
    return {
        '총 수익률':    f'{total_ret*100:.2f}%',
        '연환산 수익률': f'{annual_ret*100:.2f}%',
        'Sharpe':      f'{sharpe:.3f}',
        'MDD':         f'{max_dd*100:.2f}%',
    }


print('백테스트 실행 중...')
ppo_values = run_backtest(test_env, ppo)
ew_values  = equal_weight_backtest(test_env)

print('\n[PPO 포트폴리오]')
for k, v in compute_perf(ppo_values).items():
    print(f'  {k}: {v}')
print('\n[등가중 벤치마크]')
for k, v in compute_perf(ew_values).items():
    print(f'  {k}: {v}')

## 6. 시각화

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# 누적 수익 곡선
ax = axes[0]
ax.plot(ppo_values, label='PPO (Chronos+MTGNN)', color='mediumpurple', lw=2)
ax.plot(ew_values,  label='등가중 벤치마크',      color='steelblue',   lw=1.5, ls='--')
ax.axhline(1, color='gray', lw=0.8, ls=':')
ax.set_title(f'누적 포트폴리오 가치 (테스트: {RL_TEST_START} ~)', fontsize=13)
ax.set_ylabel('포트폴리오 가치 (초기=1.0)')
ax.set_xlabel('30분 스텝')
ax.legend()
ax.grid(alpha=0.3)

# 드로다운 곡선
ax2 = axes[1]
for vals, label, color in [
    (ppo_values, 'PPO (Chronos+MTGNN)', 'mediumpurple'),
    (ew_values,  '등가중',              'steelblue'),
]:
    dd = (vals / np.maximum.accumulate(vals)) - 1
    ax2.fill_between(range(len(dd)), dd, 0, alpha=0.3, color=color, label=label)
    ax2.plot(dd, color=color, lw=1)

ax2.set_title('드로다운 (Drawdown)', fontsize=13)
ax2.set_ylabel('Drawdown')
ax2.set_xlabel('30분 스텝')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(CRYPTO_DIR / 'crypto_rl_backtest.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 성과 요약 저장
perf_df = pd.DataFrame({
    'PPO (Chronos+MTGNN)': compute_perf(ppo_values),
    '등가중 벤치마크':      compute_perf(ew_values),
}).T

print(perf_df.to_string())
perf_df.to_csv(CRYPTO_DIR / 'crypto_rl_performance.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_rl_performance.csv, crypto_rl_backtest.png')
print()
print('▶ 다음 단계: live_trading_loop_binance.py의 PPOPolicyAdapter 구현 (crypto_ppo_portfolio.zip 로드)')
print('  또는 하이퍼파라미터 튜닝 / DTW 클러스터링 도입 검토')